In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import warnings

# Ignore routine warnings for cleaner output
warnings.filterwarnings('ignore')

# ---------------- Font size defaults ----------------
FS_TITLE   = 26   # suptitle
FS_AXLABEL = 22   # x/y axis labels
FS_TICKS   = 18   # x/y tick labels
FS_LEGEND  = 20   # legend text
FS_SUBTITLE = 22  # per-panel title

# 1. Load your dataset
print("--- 📂 Loading dataset ---")
try:
    df_new = pd.read_csv("/explore/nobackup/people/spotter5/anna_v/v2/v2_model_training_final.csv")
    df_new = df_new[df_new['flux_method'] == 'EC']

    # --- UNITS CONVERSION and FEATURE ENGINEERING ---
    df_new['tmean_C'] = df_new[['tmmn', 'tmmx']].mean(axis=1)
    df_new['date'] = pd.to_datetime(df_new[['year', 'month']].assign(day=1))
    
except FileNotFoundError as e:
    print(f"Error: The data file was not found.\n{e}")
    raise
except KeyError as e:
    print(f"Error: A required column is missing: {e}")
    raise

# 2. Define output path and get site list
comparison_plot_path = os.path.join(
    "/explore/nobackup/people/spotter5/anna_v/v2/exploration", 
    "variable_comparisons_methane"
)
os.makedirs(comparison_plot_path, exist_ok=True)
print(f"Plots will be saved to: {comparison_plot_path}")

all_sites = df_new['site_reference'].dropna().unique()
print(f"Found {len(all_sites)} unique sites to process.")

# # --- helper plotting function ---
# def create_comparison_plot(ax, site_data, var_name, var_unit, var_color):
#     ax_twin = ax.twinx()

#     # CH4 on primary axis
#     line_ch4 = ax.plot(
#         site_data['date'], site_data['ch4_flux_total'],
#         color='lightgray', label='CH4', linewidth=2.5, zorder=1
#     )
#     ax.set_ylabel('CH4', color='gray', fontsize=FS_AXLABEL)
#     ax.tick_params(axis='y', labelcolor='gray', labelsize=FS_TICKS)
#     ax.grid(True, linestyle='--', alpha=0.5)

#     # Variable on secondary axis
#     line_var = ax_twin.plot(
#         site_data['date'], site_data[var_name],
#         color=var_color, linestyle='-', label=var_name, zorder=3
#     )
#     ax_twin.set_ylabel(f'{var_name} ({var_unit})', color=var_color, fontsize=FS_AXLABEL)
#     ax_twin.tick_params(axis='y', labelcolor=var_color, labelsize=FS_TICKS)

#     ax.set_title(f'CH4 vs. {var_name}', fontsize=FS_SUBTITLE, fontweight='bold')

#     # Combined legend
#     all_handles = line_ch4 + line_var
#     all_labels = [h.get_label() for h in all_handles]
#     ax.legend(all_handles, all_labels, loc='upper left', fontsize=FS_LEGEND)


import numpy as np

def create_comparison_plot(ax, site_data, var_name, var_unit, var_color):
    ax_twin = ax.twinx()

    # ---------- CH4 on primary axis (only where data exist) ----------
    ch4_dates = site_data['date'].to_numpy()
    ch4_vals  = np.ma.masked_invalid(site_data['ch4_flux_total'].to_numpy())

    ch4_plotted = False
    if ch4_vals.count() > 0:  # at least one non-missing
        line_ch4 = ax.plot(
            ch4_dates, ch4_vals,
            color='lightgray', linewidth=2.2, marker='o', markersize=3.5,
            label='CH4', zorder=1
        )
        ch4_plotted = True
    else:
        line_ch4 = []

    ax.set_ylabel('CH4', color='gray', fontsize=FS_AXLABEL)
    ax.tick_params(axis='y', labelcolor='gray', labelsize=FS_TICKS)
    ax.grid(True, linestyle='--', alpha=0.5)

    # ---------- Variable on secondary axis (only where data exist) ----------
    if var_name not in site_data.columns:
        ax.text(0.5, 0.5, f"Missing: {var_name}", ha='center', va='center', fontsize=FS_SUBTITLE)
        ax.set_axis_off()
        return

    var_dates = site_data['date'].to_numpy()
    var_vals  = np.ma.masked_invalid(site_data[var_name].to_numpy())

    var_plotted = False
    if var_vals.count() > 0:  # at least one non-missing
        line_var = ax_twin.plot(
            var_dates, var_vals,
            color=var_color, linestyle='-', linewidth=2.2,
            marker='o', markersize=3.5, label=var_name, zorder=3
        )
        var_plotted = True
    else:
        line_var = []

    ax_twin.set_ylabel(f'{var_name} ({var_unit})', color=var_color, fontsize=FS_AXLABEL)
    ax_twin.tick_params(axis='y', labelcolor=var_color, labelsize=FS_TICKS)

    # ---------- Title & legend ----------
    ax.set_title(f'CH4 vs. {var_name}', fontsize=FS_SUBTITLE, fontweight='bold')

    handles = []
    if ch4_plotted: handles += line_ch4
    if var_plotted: handles += line_var
    if handles:
        labels = [h.get_label() for h in handles]
        ax.legend(handles, labels, loc='upper left', fontsize=FS_LEGEND)
    else:
        # nothing to show on either axis
        ax.text(0.5, 0.5, "No data", ha='center', va='center', fontsize=FS_SUBTITLE)


# --- variables to plot ---
plots = [
    ('NDVI',        'unitless',  'forestgreen'),
    ('pr',          'mm',        'dodgerblue'),
    ('tmean_C',     '°C',        'indianred'),
    ('snow_depth',  'meters',    'saddlebrown'),
    ('sm_surface',  'Volumetric','purple'),
    ('sm_rootzone', 'Volumetric','darkorange'),
]

# --- plotting loop ---
for site in all_sites:
    site_df = (
        df_new[df_new['site_reference'] == site]
        .dropna(subset=['ch4_flux_total'])
        .sort_values('date')
    )

    if site_df.empty:
        print(f" -> Skipping site: {site} (No valid CH4 observations)")
        continue

    print(f" -> Processing site: {site}")

    # 3x2 figure
    fig, axes = plt.subplots(3, 2, figsize=(30, 20), sharex=True)
    fig.suptitle(f"Site: {site} - Variable Comparison with CH4",
                 fontsize=FS_TITLE, fontweight='bold')

    ax_list = axes.flat
    for ax, (vname, vunit, vcolor) in zip(ax_list, plots):
        if vname not in site_df.columns:
            ax.text(0.5, 0.5, f"Missing: {vname}", ha='center', va='center', fontsize=FS_SUBTITLE)
            ax.set_axis_off()
            continue
        create_comparison_plot(ax, site_df, vname, vunit, vcolor)

    # common x-axis formatting
    for ax in ax_list:
        ax.set_xlabel('Date', fontsize=FS_AXLABEL)
        ax.tick_params(axis='x', labelsize=FS_TICKS)

    fig.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(os.path.join(comparison_plot_path, f"{site}.png"), dpi=150)
    plt.close(fig)

print("\n--- ✅ Finished plotting. All site comparison plots are saved. ---")


--- 📂 Loading dataset ---
Plots will be saved to: /explore/nobackup/people/spotter5/anna_v/v2/exploration/variable_comparisons_methane
Found 188 unique sites to process.
 -> Skipping site: Skyttorp 2_SE-Sk2_tower (No valid CH4 observations)
 -> Skipping site: Wolf_creek_forest_CA-WCF_tower (No valid CH4 observations)
 -> Skipping site: Alberta - Western Peatland - LaBiche River,Black Spruce,Larch Fen_CA-WP1_tower (No valid CH4 observations)
 -> Skipping site: Elgeeii forest station_RU-Ege_tower (No valid CH4 observations)
 -> Skipping site: Faejemyr_SE-Faj_tower (No valid CH4 observations)
 -> Processing site: Fyodorovskoye2_RU-Fy2_tower
 -> Skipping site: Fyodorovskoye_RU-Fyo_tower (No valid CH4 observations)
 -> Skipping site: Gunnarsholt_IS-Gun_tower (No valid CH4 observations)
 -> Skipping site: HJP02 Jack Pine_CA-HJP02_tower (No valid CH4 observations)
 -> Skipping site: HJP75 Jack Pine_CA-HJP75_tower (No valid CH4 observations)
 -> Skipping site: HJP94 Jack Pine_CA-HJP94_tower (N

In [4]:
import pandas as pd
df_new = pd.read_csv("/explore/nobackup/people/spotter5/anna_v/v2/v2_model_training_final.csv")

df_new['soil_moisture'].unique()

/explore/nobackup/people/spotter5/temp_dir/ipykernel_1299617/495248132.py:2: DtypeWarning: Columns (135) have mixed types. Specify dtype option on import or set low_memory=False.
  df_new = pd.read_csv("/explore/nobackup/people/spotter5/anna_v/v2/v2_model_training_final.csv")


array([        nan, 19.43      ,  8.53309677, ..., 20.419     ,
       20.208     , 21.308     ])

Now make heat maps

In [9]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Correlation heatmaps: predictors vs ch4_flux_total

- Monthly (per site): rows = months 1..12, cols = predictors
- Seasonal (per site): rows = Winter/Spring/Summer/Autumn, cols = predictors
- Annual (ALL SITES TOGETHER): rows = sites, cols = predictors

Outputs (under OUT_DIR):
  /monthly/<site>_corr_monthly.csv
  /monthly/<site>_corr_heatmap_monthly.png
  /seasonal/<site>_corr_seasonal.csv
  /seasonal/<site>_corr_heatmap_seasonal.png
  /annual/annual_corr_by_site.csv
  /annual/annual_corr_heatmap_by_site.png
"""

import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

# ----------------- Config -----------------
IN_CSV  = "/explore/nobackup/people/spotter5/anna_v/v2/v2_model_training_final.csv"
OUT_DIR = "/explore/nobackup/people/spotter5/anna_v/v2/corr_heatmaps_ch4"

PREDICTORS = [
    'EVI', 'NDVI', 'sur_refl_b01', 'sur_refl_b02', 'sur_refl_b03',
    'sur_refl_b07', 'NDWI', 'pdsi', 'srad', 'tmean_C', 'vap', 'vs',
    'co2_cont', 'ALT',
    'lai', 'fpar', 'Percent_NonTree_Vegetation',
    'Percent_NonVegetated', 'Percent_Tree_Cover',
    'sm_surface', 'sm_rootzone', 'snow_cover', 'snow_depth'
]

TARGET = "ch4_flux_total"
MIN_PAIRS = 3       # min (x,y) pairs required for Pearson r
ROT_X = 90           # rotate predictor labels for readability

# ----------------- Helpers -----------------
def corr_with_min_pairs(x: pd.Series, y: pd.Series, min_pairs=MIN_PAIRS) -> float:
    v = pd.DataFrame({"x": x, "y": y}).dropna()
    if len(v) < min_pairs:
        return np.nan
    return v["x"].corr(v["y"])

def season_label(month: int) -> str:
    if month in (12, 1, 2):   return "Winter"
    if month in (3, 4, 5):    return "Spring"
    if month in (6, 7, 8):    return "Summer"
    return "Autumn"

def ensure_numeric(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def plot_heatmap_annotated(matrix_df: pd.DataFrame, title: str, out_file: Path, annotate=True):
    """Blue→Red heatmap with optional r annotations in each cell."""
    n_rows, n_cols = matrix_df.shape
    fig_w = max(10, min(3 + 0.5 * n_cols, 40))
    fig_h = max(6,  2 + 0.45 * n_rows)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    im = ax.imshow(matrix_df.values, cmap="bwr", vmin=-1, vmax=1,
                   origin="upper", aspect="auto")

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Pearson r (predictor vs CH4)", fontsize=11)

    ax.set_xticks(np.arange(n_cols))
    ax.set_yticks(np.arange(n_rows))
    ax.set_xticklabels(matrix_df.columns, fontsize=11, rotation=ROT_X, ha="right")
    ax.set_yticklabels(matrix_df.index, fontsize=12)

    ax.set_xlabel("Predictor variable", fontsize=12)
    ax.set_ylabel("Group", fontsize=12)
    ax.set_title(title, fontsize=14, pad=12)

    # annotate
    if annotate:
        # turn off annotations if matrix is huge to keep files readable
        if n_rows * n_cols > 2200:
            annotate = False
        # smaller text if large but still within limit
        fz = 9 if n_rows <= 40 else 7

    if annotate:
        for i in range(n_rows):
            for j in range(n_cols):
                val = matrix_df.iloc[i, j]
                if pd.notna(val):
                    ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                            color="black", fontsize=fz)

    fig.tight_layout()
    out_file.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_file, dpi=200)
    plt.close(fig)

# ----------------- Main -----------------
def main():
    print(f"Reading: {IN_CSV}")
    df = pd.read_csv(IN_CSV)

    if "flux_method" in df.columns:
        df = df[df["flux_method"] == "EC"].copy()

    required = ["site_reference", "year", "month", TARGET]
    missing_req = [c for c in required if c not in df.columns]
    if missing_req:
        raise ValueError(f"Missing required columns: {missing_req}")

    df["year"]  = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
    df["month"] = pd.to_numeric(df["month"], errors="coerce").astype("Int64")
    df = df.dropna(subset=["site_reference", "year", "month"])
    df = df[(df["month"] >= 1) & (df["month"] <= 12)].copy()

    # keep predictors that actually exist
    preds_present = [p for p in PREDICTORS if p in df.columns]
    if not preds_present:
        raise ValueError("None of the specified predictors are present in the input CSV.")
    if len(preds_present) < len(PREDICTORS):
        print("Warning: missing predictors (ignored):",
              sorted(set(PREDICTORS) - set(preds_present)))

    df = ensure_numeric(df, [TARGET] + preds_present)
    df["season"] = df["month"].apply(season_label)

    # sites
    sites = df["site_reference"].dropna().unique()
    sites = sorted(sites, key=lambda x: str(x).lower())
    print(f"Found {len(sites)} sites.")

    # output dirs
    out_monthly  = Path(OUT_DIR) / "monthly"
    out_seasonal = Path(OUT_DIR) / "seasonal"
    out_annual   = Path(OUT_DIR) / "annual"
    for p in (out_monthly, out_seasonal, out_annual):
        p.mkdir(parents=True, exist_ok=True)

    # ---- Per-site monthly & seasonal ----
    for site in sites:
        sd = df[df["site_reference"] == site].copy()

        if sd[TARGET].dropna().shape[0] < MIN_PAIRS:
            print(f" -> Skipping {site}: insufficient {TARGET} data.")
            continue

        # Monthly (12 x predictors)
        months = list(range(1, 13))
        mon_mat = pd.DataFrame(index=months, columns=preds_present, dtype=float)
        for m in months:
            sub = sd[sd["month"] == m]
            for p in preds_present:
                mon_mat.loc[m, p] = corr_with_min_pairs(sub[p], sub[TARGET])
        mon_mat.index.name = "Month"
        mon_csv = out_monthly / f"{site}_corr_monthly.csv"
        mon_png = out_monthly / f"{site}_corr_heatmap_monthly.png"
        mon_mat.to_csv(mon_csv, float_format="%.4f")
        plot_heatmap_annotated(mon_mat, f"{site} — Monthly correlation (Predictors vs CH4)", mon_png)

        # Seasonal (4 x predictors)
        seasons = ["Winter", "Spring", "Summer", "Autumn"]
        sea_mat = pd.DataFrame(index=seasons, columns=preds_present, dtype=float)
        for s in seasons:
            sub = sd[sd["season"] == s]
            for p in preds_present:
                sea_mat.loc[s, p] = corr_with_min_pairs(sub[p], sub[TARGET])
        sea_mat.index.name = "Season"
        sea_csv = out_seasonal / f"{site}_corr_seasonal.csv"
        sea_png = out_seasonal / f"{site}_corr_heatmap_seasonal.png"
        sea_mat.to_csv(sea_csv, float_format="%.4f")
        plot_heatmap_annotated(sea_mat, f"{site} — Seasonal correlation (Predictors vs CH4)", sea_png)

        print(f" -> Saved monthly/seasonal heatmaps for {site}")

    # ---- Annual (ALL SITES TOGETHER): rows = sites, cols = predictors ----
    ann_rows = []
    valid_sites = []
    for site in sites:
        sd = df[df["site_reference"] == site].copy()
        if sd[TARGET].dropna().shape[0] < MIN_PAIRS:
            continue
        row = {}
        for p in preds_present:
            row[p] = corr_with_min_pairs(sd[p], sd[TARGET])
        ann_rows.append(row)
        valid_sites.append(site)

    if ann_rows:
        ann_mat = pd.DataFrame(ann_rows, index=valid_sites, columns=preds_present)
        ann_mat.index.name = "site_reference"
        ann_csv = out_annual / "annual_corr_by_site.csv"
        ann_png = out_annual / "annual_corr_heatmap_by_site.png"
        ann_mat.to_csv(ann_csv, float_format="%.4f")

        # annotate only if not too huge
        annotate = (ann_mat.shape[0] * ann_mat.shape[1] <= 2200)
        plot_heatmap_annotated(
            ann_mat,
            "Annual correlation (Predictors vs CH4) — rows = sites",
            ann_png,
            annotate=annotate
        )
        print(f" -> Saved combined annual matrix: {ann_csv}")
    else:
        print("No sites had sufficient data for the annual matrix.")

    print(f"\nDone. Results under:\n  {out_monthly}\n  {out_seasonal}\n  {out_annual}")

if __name__ == "__main__":
    main()


Reading: /explore/nobackup/people/spotter5/anna_v/v2/v2_model_training_final.csv
Found 188 sites.
 -> Skipping Abisko Stordalen birch forest_tower: insufficient ch4_flux_total data.
 -> Skipping Adventdalen_SJ-Adv_tower: insufficient ch4_flux_total data.
 -> Skipping Alberta - Western Peatland - LaBiche River,Black Spruce,Larch Fen_CA-WP1_tower: insufficient ch4_flux_total data.
 -> Skipping Alberta - Western Peatland - Poor Fen (Sphagnum moss)_CA-WP2_tower: insufficient ch4_flux_total data.
 -> Skipping Alberta - Western Peatland - Rich Fen  (Carex)_CA-WP3_tower: insufficient ch4_flux_total data.
 -> Skipping Anaktuvuk River Moderate Burn_US-An2_tower: insufficient ch4_flux_total data.
 -> Skipping Anaktuvuk River Severe Burn_US-An1_tower: insufficient ch4_flux_total data.
 -> Skipping Anaktuvuk River Unburned_US-An3_tower: insufficient ch4_flux_total data.
 -> Skipping Andoya_NO-And_tower: insufficient ch4_flux_total data.
 -> Saved monthly/seasonal heatmaps for ARM-NSA-Barrow_US-A10

In [10]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Correlation heatmaps: predictors vs ch4_flux_total

- Monthly (per site): rows = months 1..12, cols = predictors
- Seasonal (per site): rows = Winter/Spring/Summer/Autumn, cols = predictors
- Annual (ALL SITES TOGETHER): rows = sites, cols = predictors

Outputs (under OUT_DIR):
  /monthly/<site>_corr_monthly.csv
  /monthly/<site>_corr_heatmap_monthly.png
  /seasonal/<site>_corr_seasonal.csv
  /seasonal/<site>_corr_heatmap_seasonal.png
  /annual/annual_corr_by_site.csv
  /annual/annual_corr_heatmap_by_site.png
"""

import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

# ----------------- Config -----------------
IN_CSV  = "/explore/nobackup/people/spotter5/anna_v/v2/v2_model_training_final.csv"
OUT_DIR = "/explore/nobackup/people/spotter5/anna_v/v2/corr_heatmaps_ch4"

PREDICTORS = [
    'EVI', 'NDVI', 'sur_refl_b01', 'sur_refl_b02', 'sur_refl_b03',
    'sur_refl_b07', 'NDWI', 'pdsi', 'srad', 'tmean_C', 'vap', 'vs',
    'co2_cont', 'ALT',
    'lai', 'fpar', 'Percent_NonTree_Vegetation',
    'Percent_NonVegetated', 'Percent_Tree_Cover',
    'sm_surface', 'sm_rootzone', 'snow_cover', 'snow_depth'
]

TARGET = "ch4_flux_total"
MIN_PAIRS = 3       # min (x,y) pairs required for Pearson r
ROT_X = 90          # rotate predictor labels for readability

# ----------------- Helpers -----------------
def corr_with_min_pairs(x: pd.Series, y: pd.Series, min_pairs=MIN_PAIRS) -> float:
    v = pd.DataFrame({"x": x, "y": y}).dropna()
    if len(v) < min_pairs:
        return np.nan
    return v["x"].corr(v["y"])

def season_label(month: int) -> str:
    if month in (12, 1, 2):   return "Winter"
    if month in (3, 4, 5):    return "Spring"
    if month in (6, 7, 8):    return "Summer"
    return "Autumn"

def ensure_numeric(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def all_nan_matrix(df: pd.DataFrame) -> bool:
    """True if every cell in the matrix is NaN (i.e., no valid correlations)."""
    return df.isna().all().all()

def plot_heatmap_annotated(matrix_df: pd.DataFrame, title: str, out_file: Path, annotate=True):
    """Blue→Red heatmap with optional r annotations in each cell."""
    n_rows, n_cols = matrix_df.shape
    fig_w = max(10, min(3 + 0.5 * n_cols, 40))
    fig_h = max(6,  2 + 0.45 * n_rows)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    im = ax.imshow(matrix_df.values, cmap="bwr", vmin=-1, vmax=1,
                   origin="upper", aspect="auto")

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Pearson r (predictor vs CH4)", fontsize=11)

    ax.set_xticks(np.arange(n_cols))
    ax.set_yticks(np.arange(n_rows))
    ax.set_xticklabels(matrix_df.columns, fontsize=11, rotation=ROT_X, ha="right")
    ax.set_yticklabels(matrix_df.index, fontsize=12)

    ax.set_xlabel("Predictor variable", fontsize=12)
    ax.set_ylabel("Group", fontsize=12)
    ax.set_title(title, fontsize=14, pad=12)

    # annotate
    if annotate:
        if n_rows * n_cols > 2200:
            annotate = False
        fz = 9 if n_rows <= 40 else 7

    if annotate:
        for i in range(n_rows):
            for j in range(n_cols):
                val = matrix_df.iloc[i, j]
                if pd.notna(val):
                    ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                            color="black", fontsize=fz)

    fig.tight_layout()
    out_file.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_file, dpi=200)
    plt.close(fig)

# ----------------- Main -----------------
def main():
    print(f"Reading: {IN_CSV}")
    df = pd.read_csv(IN_CSV)

    if "flux_method" in df.columns:
        df = df[df["flux_method"] == "EC"].copy()

    required = ["site_reference", "year", "month", TARGET]
    missing_req = [c for c in required if c not in df.columns]
    if missing_req:
        raise ValueError(f"Missing required columns: {missing_req}")

    df["year"]  = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
    df["month"] = pd.to_numeric(df["month"], errors="coerce").astype("Int64")
    df = df.dropna(subset=["site_reference", "year", "month"])
    df = df[(df["month"] >= 1) & (df["month"] <= 12)].copy()

    # keep predictors that actually exist
    preds_present = [p for p in PREDICTORS if p in df.columns]
    if not preds_present:
        raise ValueError("None of the specified predictors are present in the input CSV.")
    if len(preds_present) < len(PREDICTORS):
        print("Warning: missing predictors (ignored):",
              sorted(set(PREDICTORS) - set(preds_present)))

    df = ensure_numeric(df, [TARGET] + preds_present)
    df["season"] = df["month"].apply(season_label)

    # sites
    sites = df["site_reference"].dropna().unique()
    sites = sorted(sites, key=lambda x: str(x).lower())
    print(f"Found {len(sites)} sites.")

    # output dirs
    out_monthly  = Path(OUT_DIR) / "monthly"
    out_seasonal = Path(OUT_DIR) / "seasonal"
    out_annual   = Path(OUT_DIR) / "annual"
    for p in (out_monthly, out_seasonal, out_annual):
        p.mkdir(parents=True, exist_ok=True)

    # ---- Per-site monthly & seasonal ----
    for site in sites:
        sd = df[df["site_reference"] == site].copy()

        if sd[TARGET].dropna().shape[0] < MIN_PAIRS:
            print(f" -> Skipping {site}: insufficient {TARGET} data.")
            continue

        # Monthly (12 x predictors)
        months = list(range(1, 13))
        mon_mat = pd.DataFrame(index=months, columns=preds_present, dtype=float)
        for m in months:
            sub = sd[sd["month"] == m]
            for p in preds_present:
                mon_mat.loc[m, p] = corr_with_min_pairs(sub[p], sub[TARGET])
        mon_mat.index.name = "Month"

        # Seasonal (4 x predictors)
        seasons = ["Winter", "Spring", "Summer", "Autumn"]
        sea_mat = pd.DataFrame(index=seasons, columns=preds_present, dtype=float)
        for s in seasons:
            sub = sd[sd["season"] == s]
            for p in preds_present:
                sea_mat.loc[s, p] = corr_with_min_pairs(sub[p], sub[TARGET])
        sea_mat.index.name = "Season"

        # ---- NEW: Skip saving this site if *both* matrices are entirely NaN ----
        mon_all_nan = all_nan_matrix(mon_mat)
        sea_all_nan = all_nan_matrix(sea_mat)
        if mon_all_nan and sea_all_nan:
            print(f" -> Skipping {site}: all correlations are NaN in monthly and seasonal matrices.")
            continue

        # Save/plot only the matrices that are not entirely NaN
        if not mon_all_nan:
            mon_csv = out_monthly / f"{site}_corr_monthly.csv"
            mon_png = out_monthly / f"{site}_corr_heatmap_monthly.png"
            mon_mat.to_csv(mon_csv, float_format="%.4f")
            plot_heatmap_annotated(mon_mat, f"{site} — Monthly correlation (Predictors vs CH4)", mon_png)
        else:
            print(f"    (No monthly output for {site}: monthly matrix is all NaN)")

        if not sea_all_nan:
            sea_csv = out_seasonal / f"{site}_corr_seasonal.csv"
            sea_png = out_seasonal / f"{site}_corr_heatmap_seasonal.png"
            sea_mat.to_csv(sea_csv, float_format="%.4f")
            plot_heatmap_annotated(sea_mat, f"{site} — Seasonal correlation (Predictors vs CH4)", sea_png)
        else:
            print(f"    (No seasonal output for {site}: seasonal matrix is all NaN)")

        if not (mon_all_nan and sea_all_nan):
            print(f" -> Saved monthly/seasonal outputs for {site}")

    # ---- Annual (ALL SITES TOGETHER): rows = sites, cols = predictors ----
    ann_rows = []
    valid_sites = []
    for site in sites:
        sd = df[df["site_reference"] == site].copy()
        if sd[TARGET].dropna().shape[0] < MIN_PAIRS:
            continue
        row = {}
        for p in preds_present:
            row[p] = corr_with_min_pairs(sd[p], sd[TARGET])
        # ---- NEW: Only keep this site if at least one correlation is not NaN ----
        if pd.Series(row).notna().any():
            ann_rows.append(row)
            valid_sites.append(site)

    if ann_rows:
        ann_mat = pd.DataFrame(ann_rows, index=valid_sites, columns=preds_present)
        ann_mat.index.name = "site_reference"
        ann_csv = out_annual / "annual_corr_by_site.csv"
        ann_png = out_annual / "annual_corr_heatmap_by_site.png"
        ann_mat.to_csv(ann_csv, float_format="%.4f")

        annotate = (ann_mat.shape[0] * ann_mat.shape[1] <= 2200)
        plot_heatmap_annotated(
            ann_mat,
            "Annual correlation (Predictors vs CH4) — rows = sites",
            ann_png,
            annotate=annotate
        )
        print(f" -> Saved combined annual matrix: {ann_csv}")
    else:
        print("No sites had any valid (non-NaN) annual correlations.")

    print(f"\nDone. Results under:\n  {out_monthly}\n  {out_seasonal}\n  {out_annual}")

if __name__ == "__main__":
    main()


Reading: /explore/nobackup/people/spotter5/anna_v/v2/v2_model_training_final.csv
Found 188 sites.
 -> Skipping Abisko Stordalen birch forest_tower: insufficient ch4_flux_total data.
 -> Skipping Adventdalen_SJ-Adv_tower: insufficient ch4_flux_total data.
 -> Skipping Alberta - Western Peatland - LaBiche River,Black Spruce,Larch Fen_CA-WP1_tower: insufficient ch4_flux_total data.
 -> Skipping Alberta - Western Peatland - Poor Fen (Sphagnum moss)_CA-WP2_tower: insufficient ch4_flux_total data.
 -> Skipping Alberta - Western Peatland - Rich Fen  (Carex)_CA-WP3_tower: insufficient ch4_flux_total data.
 -> Skipping Anaktuvuk River Moderate Burn_US-An2_tower: insufficient ch4_flux_total data.
 -> Skipping Anaktuvuk River Severe Burn_US-An1_tower: insufficient ch4_flux_total data.
 -> Skipping Anaktuvuk River Unburned_US-An3_tower: insufficient ch4_flux_total data.
 -> Skipping Andoya_NO-And_tower: insufficient ch4_flux_total data.
 -> Saved monthly/seasonal outputs for ARM-NSA-Barrow_US-A10_

Soil correlations across sites

In [6]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Correlations across ALL SITES between soil variables and CH4 flux.

- Soil vars are static (one value per site_reference).
- We aggregate CH4 per site (mean over all records for that site).
- Then compute Pearson r across sites: soil_var vs site-mean CH4.

Outputs (under OUT_DIR):
  /soils/soil_corr_across_sites.csv
  /soils/soil_corr_across_sites.png
"""

import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

# ----------------- Config -----------------
IN_CSV  = "/explore/nobackup/people/spotter5/anna_v/v2/v2_model_training_final.csv"
OUT_DIR = "/explore/nobackup/people/spotter5/anna_v/v2/corr_heatmaps_ch4"
TARGET  = "ch4_flux_total"
MIN_SITES = 3         # min number of sites required to compute a correlation
ROT_X   = 60          # rotation for soil var labels

SOIL_VARS = [
    "bdod_0_100cm",
    "cec_0_100cm",
    "cfvo_0_100cm",
    "clay_0_100cm",
    "nitrogen_0_100cm",
    "ocd_0_100cm",
    "phh2o_0_100cm",
    "sand_0_100cm",
    "silt_0_100cm",
    "soc_0_100cm",
]

# ----------------- Helpers -----------------
def ensure_numeric(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def first_nonnull(series: pd.Series):
    s = series.dropna()
    return s.iloc[0] if not s.empty else np.nan

def plot_single_row_heatmap(values: pd.Series, title: str, out_file: Path):
    """Heatmap with a single row (soil vars) annotated with Pearson r."""
    mat = pd.DataFrame([values.values], index=["Pearson r"], columns=values.index)

    n_cols = mat.shape[1]
    fig_w = max(10, min(3 + 0.5 * n_cols, 40))
    fig_h = 3.5

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    im = ax.imshow(mat.values, cmap="bwr", vmin=-1, vmax=1, aspect="auto")

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Pearson r (soil vs site-mean CH4)", fontsize=11)

    ax.set_xticks(np.arange(n_cols))
    ax.set_xticklabels(mat.columns, rotation=ROT_X, ha="right", fontsize=10)
    ax.set_yticks([0])
    ax.set_yticklabels([""], fontsize=12)

    # annotate
    for j in range(n_cols):
        val = mat.iloc[0, j]
        if pd.notna(val):
            ax.text(j, 0, f"{val:.2f}", ha="center", va="center", color="black", fontsize=10)

    ax.set_title(title, fontsize=14, pad=10)
    ax.set_xlabel("Soil variable", fontsize=12)
    fig.tight_layout()
    out_file.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_file, dpi=200)
    plt.close(fig)

# ----------------- Main -----------------
def main():
    print(f"Reading: {IN_CSV}")
    df = pd.read_csv(IN_CSV, low_memory=False)

    # Optional: keep only eddy-covariance sites if column exists
    if "flux_method" in df.columns:
        df = df[df["flux_method"] == "EC"].copy()

    # Required columns
    req = ["site_reference", TARGET]
    missing = [c for c in req if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # Keep only soil variables that are present
    soil_present = [c for c in SOIL_VARS if c in df.columns]
    if not soil_present:
        raise ValueError("None of the requested soil variables are present in the input CSV.")
    if len(soil_present) < len(SOIL_VARS):
        missing_soils = sorted(set(SOIL_VARS) - set(soil_present))
        print("Warning: missing soil variables (ignored):", missing_soils)

    # Numeric casting
    df = ensure_numeric(df, [TARGET] + soil_present)

    # One row per site for soil vars (take first non-null per site)
    soil_by_site = df.groupby("site_reference")[soil_present].agg(first_nonnull)

    # Site-level CH4: mean over time (remove temporal cols entirely)
    ch4_by_site = df.groupby("site_reference")[TARGET].mean().to_frame("site_mean_ch4")

    # Join and drop rows with missing target
    site_level = soil_by_site.join(ch4_by_site, how="inner").dropna(subset=["site_mean_ch4"])

    # Compute Pearson r for each soil var vs site-mean CH4
    corrs = {}
    for var in soil_present:
        sub = site_level[[var, "site_mean_ch4"]].dropna()
        if len(sub) >= MIN_SITES:
            corrs[var] = sub[var].corr(sub["site_mean_ch4"])
        else:
            corrs[var] = np.nan

    corr_series = pd.Series(corrs).sort_index()

    # Outputs
    out_soils = Path(OUT_DIR) / "soils"
    out_soils.mkdir(parents=True, exist_ok=True)

    csv_path = out_soils / "soil_corr_across_sites.csv"
    png_path = out_soils / "soil_corr_across_sites.png"

    corr_series.to_frame("pearson_r").to_csv(csv_path, float_format="%.4f")
    plot_single_row_heatmap(corr_series, "Soil vs Site-mean CH4 (Across Sites)", png_path)

    n_sites = len(site_level)
    print(f"Computed correlations using {n_sites} site(s).")
    print(f"Saved:\n  {csv_path}\n  {png_path}")

if __name__ == "__main__":
    main()


Reading: /explore/nobackup/people/spotter5/anna_v/v2/v2_model_training_final.csv
Computed correlations using 69 site(s).
Saved:
  /explore/nobackup/people/spotter5/anna_v/v2/corr_heatmaps_ch4/soils/soil_corr_across_sites.csv
  /explore/nobackup/people/spotter5/anna_v/v2/corr_heatmaps_ch4/soils/soil_corr_across_sites.png


Summer ch4

In [17]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Correlations across ALL SITES between soil variables and **mean summer (JJA) CH4**.

- Soil vars are static (one value per site_reference).
- We compute per-site mean CH4 using ONLY June/July/August observations across all years.
- Sites without any JJA CH4 observations are excluded.
- Then compute Pearson r across sites: soil_var vs site-mean-summer CH4.

Outputs (under OUT_DIR):
  /soils/soil_corr_across_sites_JJA.csv
  /soils/soil_corr_across_sites_JJA.png
"""

import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

# ----------------- Config -----------------
IN_CSV  = "/explore/nobackup/people/spotter5/anna_v/v2/v2_model_training_final.csv"
OUT_DIR = "/explore/nobackup/people/spotter5/anna_v/v2/corr_heatmaps_ch4"
TARGET  = "ch4_flux_total"
MIN_SITES = 3         # min number of sites required to compute a correlation
ROT_X   = 60          # rotation for soil var labels

SOIL_VARS = [
    "bdod_0_100cm",
    "cec_0_100cm",
    "cfvo_0_100cm",
    "clay_0_100cm",
    "nitrogen_0_100cm",
    "ocd_0_100cm",
    "phh2o_0_100cm",
    "sand_0_100cm",
    "silt_0_100cm",
    "soc_0_100cm",
]

# ----------------- Helpers -----------------
def ensure_numeric(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def first_nonnull(series: pd.Series):
    s = series.dropna()
    return s.iloc[0] if not s.empty else np.nan

def plot_single_row_heatmap(values: pd.Series, title: str, out_file: Path):
    """Heatmap with a single row (soil vars) annotated with Pearson r."""
    mat = pd.DataFrame([values.values], index=["Pearson r"], columns=values.index)

    n_cols = mat.shape[1]
    fig_w = max(10, min(3 + 0.5 * n_cols, 40))
    fig_h = 3.5

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    im = ax.imshow(mat.values, cmap="bwr", vmin=-1, vmax=1, aspect="auto")

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Pearson r (soil vs site-mean summer CH4)", fontsize=11)

    ax.set_xticks(np.arange(n_cols))
    ax.set_xticklabels(mat.columns, rotation=ROT_X, ha="right", fontsize=10)
    ax.set_yticks([0])
    ax.set_yticklabels([""], fontsize=12)

    # annotate
    for j in range(n_cols):
        val = mat.iloc[0, j]
        if pd.notna(val):
            ax.text(j, 0, f"{val:.2f}", ha="center", va="center", color="black", fontsize=10)

    ax.set_title(title, fontsize=14, pad=10)
    ax.set_xlabel("Soil variable", fontsize=12)
    fig.tight_layout()
    out_file.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_file, dpi=200)
    plt.close(fig)

# ----------------- Main -----------------
def main():
    print(f"Reading: {IN_CSV}")
    df = pd.read_csv(IN_CSV, low_memory=False)

    # Optional: keep only eddy-covariance sites if column exists
    if "flux_method" in df.columns:
        df = df[df["flux_method"] == "EC"].copy()

    # Required columns
    req = ["site_reference", "month", TARGET]
    missing = [c for c in req if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # Keep only soil variables that are present
    soil_present = [c for c in SOIL_VARS if c in df.columns]
    if not soil_present:
        raise ValueError("None of the requested soil variables are present in the input CSV.")
    if len(soil_present) < len(SOIL_VARS):
        missing_soils = sorted(set(SOIL_VARS) - set(soil_present))
        print("Warning: missing soil variables (ignored):", missing_soils)

    # Numeric casting
    df = ensure_numeric(df, ["month", TARGET] + soil_present)

    # Restrict to summer months (June, July, August)
    df = df[df["month"].isin([6, 7, 8])].copy()

    # Sites must have at least one JJA CH4 observation
    df = df.dropna(subset=["site_reference", TARGET])

    # One row per site for soil vars (take first non-null per site)
    soil_by_site = df.groupby("site_reference")[soil_present].agg(first_nonnull)

    # Site-level CH4: mean over JJA records (across all years)
    ch4_by_site = (
        df.groupby("site_reference")[TARGET]
        .mean()
        .to_frame("site_mean_summer_ch4")
    )

    # Join and drop rows with missing target
    site_level = soil_by_site.join(ch4_by_site, how="inner").dropna(subset=["site_mean_summer_ch4"])

    # Compute Pearson r for each soil var vs site-mean-summer CH4
    corrs = {}
    for var in soil_present:
        sub = site_level[[var, "site_mean_summer_ch4"]].dropna()
        if len(sub) >= MIN_SITES:
            corrs[var] = sub[var].corr(sub["site_mean_summer_ch4"])
        else:
            corrs[var] = np.nan

    corr_series = pd.Series(corrs).sort_index()

    # Outputs
    out_soils = Path(OUT_DIR) / "soils"
    out_soils.mkdir(parents=True, exist_ok=True)

    csv_path = out_soils / "soil_corr_across_sites_JJA.csv"
    png_path = out_soils / "soil_corr_across_sites_JJA.png"

    corr_series.to_frame("pearson_r").to_csv(csv_path, float_format="%.4f")
    plot_single_row_heatmap(corr_series, "Soil vs Site-mean Summer (JJA) CH4 — Across Sites", png_path)

    n_sites = site_level.shape[0]
    print(f"Computed correlations using {n_sites} site(s) with JJA CH4.")
    print(f"Saved:\n  {csv_path}\n  {png_path}")

if __name__ == "__main__":
    main()


Reading: /explore/nobackup/people/spotter5/anna_v/v2/v2_model_training_final.csv
Computed correlations using 66 site(s) with JJA CH4.
Saved:
  /explore/nobackup/people/spotter5/anna_v/v2/corr_heatmaps_ch4/soils/soil_corr_across_sites_JJA.csv
  /explore/nobackup/people/spotter5/anna_v/v2/corr_heatmaps_ch4/soils/soil_corr_across_sites_JJA.png


site level soil moisture

In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import warnings
import numpy as np
import re

warnings.filterwarnings('ignore')

# ---------------- Font size defaults ----------------
FS_TITLE   = 26
FS_AXLABEL = 22
FS_TICKS   = 18
FS_LEGEND  = 20
FS_SUBTITLE = 22

# 1. Load dataset
print("--- 📂 Loading dataset ---")
df_new = pd.read_csv("/explore/nobackup/people/spotter5/anna_v/v2/v2_model_training_final.csv")
if 'flux_method' in df_new.columns:
    df_new = df_new[df_new['flux_method'] == 'EC']

if {'tmmn','tmmx'}.issubset(df_new.columns):
    df_new['tmean_C'] = df_new[['tmmn', 'tmmx']].mean(axis=1)
df_new['date'] = pd.to_datetime(df_new[['year', 'month']].assign(day=1), errors='coerce')

# 2. Output directory
comparison_plot_path = os.path.join(
    "/explore/nobackup/people/spotter5/anna_v/v2/exploration",
    "variable_comparisons_methane_moisture"
)
os.makedirs(comparison_plot_path, exist_ok=True)

all_sites = df_new['site_reference'].dropna().unique()
print(f"Found {len(all_sites)} unique sites to process.")

def create_comparison_plot(ax, site_data, var_name, display_name, var_unit, var_color,
                           right_ylim=None):
    """Plot CH4 on left axis, the variable on right axis; optionally set fixed right y-lims."""
    ax_twin = ax.twinx()

    ch4_dates = site_data['date'].to_numpy()
    ch4_vals  = np.ma.masked_invalid(site_data['ch4_flux_total'].to_numpy())

    ch4_plotted = False
    if ch4_vals.count() > 0:
        line_ch4 = ax.plot(
            ch4_dates, ch4_vals,
            linewidth=2.2, marker='o', markersize=3.5,
            color='lightgray', label='CH4', zorder=1
        )
        ch4_plotted = True
    else:
        line_ch4 = []

    ax.set_ylabel('CH4', color='gray', fontsize=FS_AXLABEL)
    ax.tick_params(axis='y', labelcolor='gray', labelsize=FS_TICKS)
    ax.grid(True, linestyle='--', alpha=0.5)

    if var_name not in site_data.columns:
        ax.text(0.5, 0.5, f"Missing: {var_name}", ha='center', va='center', fontsize=FS_SUBTITLE)
        ax.set_axis_off()
        return

    var_dates = site_data['date'].to_numpy()
    var_vals  = np.ma.masked_invalid(site_data[var_name].to_numpy())

    var_plotted = False
    if var_vals.count() > 0:
        line_var = ax_twin.plot(
            var_dates, var_vals,
            color=var_color, linestyle='-', linewidth=2.2,
            marker='o', markersize=3.5, label=display_name, zorder=3
        )
        var_plotted = True
    else:
        line_var = []

    ax_twin.set_ylabel(f'{display_name} ({var_unit})', color=var_color, fontsize=FS_AXLABEL)
    ax_twin.tick_params(axis='y', labelcolor=var_color, labelsize=FS_TICKS)

    if right_ylim is not None and np.isfinite(right_ylim).all():
        ax_twin.set_ylim(right_ylim)

    ax.set_title(f'CH4 vs. {display_name}', fontsize=FS_SUBTITLE, fontweight='bold')

    handles = []
    if ch4_plotted: handles += line_ch4
    if var_plotted: handles += line_var
    if handles:
        labels = [h.get_label() for h in handles]
        ax.legend(handles, labels, loc='upper left', fontsize=FS_LEGEND)
    else:
        ax.text(0.5, 0.5, "No data", ha='center', va='center', fontsize=FS_SUBTITLE)


# --- variables to plot (2x2) ---
plots = [
    ('pr',            'precipitation', 'mm',         'dodgerblue'),
    ('sm_surface',    'sm_surface',    'volumetric', 'purple'),
    ('sm_rootzone',   'sm_rootzone',   'volumetric', 'darkorange'),
    ('soil_moisture', 'soil_moisture', 'volumetric', 'teal'),
]

# --- loop over sites ---
for site in all_sites:
    site_df = (
        df_new[df_new['site_reference'] == site]
        .dropna(subset=['ch4_flux_total'])
        .sort_values('date')
        .copy()
    )

    if site_df.empty:
        continue

    # Scale soil_moisture by 100 if present
    if 'soil_moisture' in site_df.columns:
        site_df['soil_moisture'] = site_df['soil_moisture'] / 100.0

    # Only plot if at least one non-NaN soil_moisture observation
    if 'soil_moisture' not in site_df.columns or site_df['soil_moisture'].dropna().empty:
        print(f" -> Skipping site: {site} (no soil_moisture data)")
        continue

    # Determine shared y-axis limits for moisture variables
    moisture_vars = [v for v in ['sm_surface', 'sm_rootzone', 'soil_moisture'] if v in site_df.columns]
    moisture_vals = []
    for v in moisture_vars:
        vals = np.asarray(site_df[v], dtype='float64')
        vals = vals[np.isfinite(vals)]
        if vals.size:
            moisture_vals.append(vals)
    right_ylim = None
    if moisture_vals:
        all_m = np.concatenate(moisture_vals)
        y_min, y_max = np.nanmin(all_m), np.nanmax(all_m)
        if np.isfinite(y_min) and np.isfinite(y_max) and y_max > y_min:
            pad = 0.05 * (y_max - y_min)
            right_ylim = (y_min - pad, y_max + pad)

    print(f" -> Processing site: {site}")

    fig, axes = plt.subplots(2, 2, figsize=(24, 16), sharex=True)
    fig.suptitle(f"Site: {site} - Variable Comparison with CH4",
                 fontsize=FS_TITLE, fontweight='bold')

    ax_list = axes.flat
    for ax, (vname, vdisp, vunit, vcolor) in zip(ax_list, plots):
        if vname not in site_df.columns:
            ax.text(0.5, 0.5, f"Missing: {vname}", ha='center', va='center', fontsize=FS_SUBTITLE)
            ax.set_axis_off()
            continue
        ylim = right_ylim if vname in ['sm_surface', 'sm_rootzone', 'soil_moisture'] else None
        create_comparison_plot(ax, site_df, vname, vdisp, vunit, vcolor, right_ylim=ylim)

    for ax in ax_list:
        ax.set_xlabel('Date', fontsize=FS_AXLABEL)
        ax.tick_params(axis='x', labelsize=FS_TICKS)

    fig.tight_layout(rect=[0, 0, 1, 0.95])

    # Safe filename
    safe_site = re.sub(r'[^A-Za-z0-9_.-]+', '_', str(site))
    out_fp = os.path.join(comparison_plot_path, f"{safe_site}_2x2.png")
    plt.savefig(out_fp, dpi=150)
    plt.close(fig)

print("\n--- ✅ Finished plotting. Only sites with soil_moisture data were plotted. ---")


--- 📂 Loading dataset ---
Found 199 unique sites to process.
 -> Skipping site: Lost Creek_US-Los_tower (no soil_moisture data)
 -> Skipping site: Marcell Bog Lake Peatland_US-MBP_tower (no soil_moisture data)
 -> Skipping site: Newdale Manitoba_CA-EM1_tower (no soil_moisture data)
 -> Processing site: Park Falls/WLEF_US-PFa_tower
 -> Skipping site: Allequash Creek Site_US-ALQ_tower (no soil_moisture data)
 -> Skipping site: Delta Burns Bog 2_CA-DB2_tower (no soil_moisture data)
 -> Skipping site: Delta Burns Bog_CA-DBB_tower (no soil_moisture data)
 -> Processing site: Fyodorovskoye2_RU-Fy2_tower
 -> Processing site: Howland Forest (main tower)_US-Ho1_tower
 -> Processing site: Pitsalu_tower
 -> Skipping site: Shoal Lake Manitoba_CA-EM2_tower (no soil_moisture data)
 -> Skipping site: Trout Lake Area South Sparkling Bog_US-TrS_tower (no soil_moisture data)
 -> Processing site: Bibai bog_JP-Bby_tower
 -> Processing site: Bonanza Creek Black Spruce_US-BZS_tower
 -> Processing site: Bona

soil temperature

In [16]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import warnings
import numpy as np
import re

warnings.filterwarnings('ignore')

# ---------------- Font size defaults ----------------
FS_TITLE   = 26
FS_AXLABEL = 22
FS_TICKS   = 18
FS_LEGEND  = 20
FS_SUBTITLE = 22

# 1. Load dataset
print("--- 📂 Loading dataset ---")
df_new = pd.read_csv("/explore/nobackup/people/spotter5/anna_v/v2/v2_model_training_final.csv")
if 'flux_method' in df_new.columns:
    df_new = df_new[df_new['flux_method'] == 'EC']

# Mean temperature field
if {'tmmn','tmmx'}.issubset(df_new.columns):
    df_new['tmean_C'] = df_new[['tmmn', 'tmmx']].mean(axis=1)
df_new['date'] = pd.to_datetime(df_new[['year', 'month']].assign(day=1), errors='coerce')

# 2. Output directory
comparison_plot_path = os.path.join(
    "/explore/nobackup/people/spotter5/anna_v/v2/exploration",
    "variable_comparisons_methane_soiltemp"
)
os.makedirs(comparison_plot_path, exist_ok=True)
print(f"Plots will be saved to: {comparison_plot_path}")

all_sites = df_new['site_reference'].dropna().unique()
print(f"Found {len(all_sites)} unique sites to process.")

# ---------- Helper: CH4 vs variable plotting ----------
def create_comparison_plot(ax, site_data, var_name, display_name, var_unit, var_color,
                           right_ylim=None):
    ax_twin = ax.twinx()

    # ---------- CH4 ----------
    ch4_dates = site_data['date'].to_numpy()
    ch4_vals  = np.ma.masked_invalid(site_data['ch4_flux_total'].to_numpy())

    ch4_plotted = False
    if ch4_vals.count() > 0:
        line_ch4 = ax.plot(
            ch4_dates, ch4_vals,
            linewidth=2.2, marker='o', markersize=3.5,
            color='lightgray', label='CH4', zorder=1
        )
        ch4_plotted = True
    else:
        line_ch4 = []

    ax.set_ylabel('CH4', color='gray', fontsize=FS_AXLABEL)
    ax.tick_params(axis='y', labelcolor='gray', labelsize=FS_TICKS)
    ax.grid(True, linestyle='--', alpha=0.5)

    # ---------- Soil temperature ----------
    if var_name not in site_data.columns:
        ax.text(0.5, 0.5, f"Missing: {var_name}", ha='center', va='center', fontsize=FS_SUBTITLE)
        ax.set_axis_off()
        return

    var_dates = site_data['date'].to_numpy()
    var_vals  = np.ma.masked_invalid(site_data[var_name].to_numpy())

    var_plotted = False
    if var_vals.count() > 0:
        line_var = ax_twin.plot(
            var_dates, var_vals,
            color=var_color, linestyle='-', linewidth=2.2,
            marker='o', markersize=3.5, label=f"Soil T {display_name}", zorder=3
        )
        var_plotted = True
    else:
        line_var = []

    ax_twin.set_ylabel(f"Soil Temperature ({var_unit})", color=var_color, fontsize=FS_AXLABEL)
    ax_twin.tick_params(axis='y', labelcolor=var_color, labelsize=FS_TICKS)

    if right_ylim is not None and np.isfinite(right_ylim).all():
        ax_twin.set_ylim(right_ylim)

    ax.set_title(f"CH4 vs. Soil T {display_name}", fontsize=FS_SUBTITLE, fontweight='bold')

    handles = []
    if ch4_plotted: handles += line_ch4
    if var_plotted: handles += line_var
    if handles:
        labels = [h.get_label() for h in handles]
        ax.legend(handles, labels, loc='upper left', fontsize=FS_LEGEND)
    else:
        ax.text(0.5, 0.5, "No data", ha='center', va='center', fontsize=FS_SUBTITLE)


# --- variables to plot (2x2, converted to °C) ---
plots = [
    ('soil_temperature_level_1', '0–7 cm',     '°C', 'dodgerblue'),
    ('soil_temperature_level_2', '7–28 cm',    '°C', 'purple'),
    ('soil_temperature_level_3', '28–100 cm',  '°C', 'darkorange'),
    ('soil_temperature_level_4', '100–289 cm', '°C', 'teal'),
]

# --- main loop ---
for site in all_sites:
    site_df = (
        df_new[df_new['site_reference'] == site]
        .sort_values('date')
        .copy()
    )
    if site_df.empty:
        continue

    # Convert soil temperatures from Kelvin to °C
    for col in ['soil_temperature_level_1', 'soil_temperature_level_2',
                'soil_temperature_level_3', 'soil_temperature_level_4']:
        if col in site_df.columns:
            site_df[col] = site_df[col] - 273.15

    # --- Skip if no CH4 or top-layer soil temp data ---
    valid_ch4 = site_df['ch4_flux_total'].notna().sum()
    valid_t1  = site_df['soil_temperature_level_1'].notna().sum() if 'soil_temperature_level_1' in site_df.columns else 0
    if valid_ch4 == 0 or valid_t1 == 0:
        print(f" -> Skipping site: {site} (no CH4 or soil_temperature_level_1 data)")
        continue

    # --- Restrict time range to first→last CH4 observation ---
    ch4_valid = site_df[site_df['ch4_flux_total'].notna()]
    if ch4_valid.empty:
        continue
    first_date, last_date = ch4_valid['date'].min(), ch4_valid['date'].max()
    site_df = site_df[(site_df['date'] >= first_date) & (site_df['date'] <= last_date)]

    # Determine shared y-axis limits for soil temperature variables
    temp_vars = [v for v, _, _, _ in plots if v in site_df.columns]
    all_vals = []
    for v in temp_vars:
        vals = np.asarray(site_df[v], dtype='float64')
        vals = vals[np.isfinite(vals)]
        if vals.size:
            all_vals.append(vals)
    right_ylim = None
    if all_vals:
        all_t = np.concatenate(all_vals)
        y_min, y_max = np.nanmin(all_t), np.nanmax(all_t)
        if np.isfinite(y_min) and np.isfinite(y_max) and y_max > y_min:
            pad = 0.05 * (y_max - y_min)
            right_ylim = (y_min - pad, y_max + pad)

    print(f" -> Processing site: {site}  |  CH4 period: {first_date.date()} → {last_date.date()}")

    fig, axes = plt.subplots(2, 2, figsize=(24, 16), sharex=True)
    fig.suptitle(f"Site: {site} – CH4 vs. Soil Temperature ({first_date.date()}–{last_date.date()})",
                 fontsize=FS_TITLE, fontweight='bold')

    ax_list = axes.flat
    for ax, (vname, vdisp, vunit, vcolor) in zip(ax_list, plots):
        if vname not in site_df.columns:
            ax.text(0.5, 0.5, f"Missing: {vname}", ha='center', va='center', fontsize=FS_SUBTITLE)
            ax.set_axis_off()
            continue
        create_comparison_plot(ax, site_df, vname, vdisp, vunit, vcolor, right_ylim=right_ylim)

    for ax in ax_list:
        ax.set_xlabel('Date', fontsize=FS_AXLABEL)
        ax.tick_params(axis='x', labelsize=FS_TICKS)

    fig.tight_layout(rect=[0, 0, 1, 0.95])

    # Safe filename
    safe_site = re.sub(r'[^A-Za-z0-9_.-]+', '_', str(site))
    out_fp = os.path.join(comparison_plot_path, f"{safe_site}_soiltemp_2x2.png")
    plt.savefig(out_fp, dpi=150)
    plt.close(fig)

print("\n--- ✅ Finished plotting soil temperature comparisons (restricted to CH4 observation period). ---")


--- 📂 Loading dataset ---
Plots will be saved to: /explore/nobackup/people/spotter5/anna_v/v2/exploration/variable_comparisons_methane_soiltemp
Found 199 unique sites to process.
 -> Processing site: Lost Creek_US-Los_tower  |  CH4 period: 2014-01-01 → 2025-04-01
 -> Processing site: Marcell Bog Lake Peatland_US-MBP_tower  |  CH4 period: 2015-01-01 → 2018-12-01
 -> Processing site: Newdale Manitoba_CA-EM1_tower  |  CH4 period: 2021-05-01 → 2023-12-01
 -> Processing site: Park Falls/WLEF_US-PFa_tower  |  CH4 period: 2010-09-01 → 2024-11-01
 -> Skipping site: Skyttorp 2_SE-Sk2_tower (no CH4 or soil_temperature_level_1 data)
 -> Skipping site: Wolf_creek_forest_CA-WCF_tower (no CH4 or soil_temperature_level_1 data)
 -> Skipping site: Alberta - Western Peatland - LaBiche River,Black Spruce,Larch Fen_CA-WP1_tower (no CH4 or soil_temperature_level_1 data)
 -> Processing site: Allequash Creek Site_US-ALQ_tower  |  CH4 period: 2019-01-01 → 2020-12-01
 -> Processing site: Delta Burns Bog 2_CA-D